# Portfolio-opdracht 2: Voorspel verhuuraantallen (CityBike)

**Deel: Opdracht 1 — Exploratieve Data Analyse (EDA)** · Khaleel Al-Aqel

Dit notebook is het eerste deel van ons portfolio. Aan het eind worden de delen samengevoegd in
`Portfolio2_CityBike.ipynb`. De data (`train.csv`, `test.csv`, `sample_submission.csv`) staat in
de map `data/`.

**Inhoud**

- 1.1 Imports en instellingen · 1.2 Data inladen · 1.3 Datatypes, basisstatistieken en verdeling
- 1.4 Datakwaliteit · 1.5 Tijdselementen toevoegen · 1.6 Patronen in de tijd
- 1.7 Decompositie: trend, seizoen en residu · 1.8 Relaties met `cnt` (juiste correlatiemaat)
- 1.9 Data opschonen · 1.10 Bevindingen van de EDA


# 1. Opdracht 1: Exploratieve Data Analyse (EDA)

## 1.1 Imports en instellingen


In [ ]:
# Bron: Week 10 – Regression Time Series 2 (import-cel)
# Bron: Week 9 – Time series analysis 1 (seasonal_decompose)

# Importeren van de packages die nodig zijn voor de EDA
import os
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from statsmodels.tsa.seasonal import seasonal_decompose

# Waarschuwingen onderdrukken voor een overzichtelijk notebook
warnings.filterwarnings('ignore')

# Vaste instellingen
DATA_FOLDER = os.path.join('..', 'data')     # map met de Kaggle-bestanden
pd.set_option('display.max_colwidth', 150)
plt.rcParams['figure.figsize'] = (12, 4)
sns.set_style('whitegrid')


## 1.2 Data inladen

De klasse `DataLoader` leest de bestanden in en zet `date_hour` direct om naar het
**DateTime-formaat** (Radhakrishnan, 2023b), met `parse_dates` zoals in het notebook
*Regression Time Series 2* (De Haagse Hogeschool, z.d.-d). Het opschonen (methode `clean`) volgt
in 1.9; die code komt niet in de lesstof voor (Claude, 2026. Prompt 1: Portfolio 2 CityBike
notebook).

In [ ]:
# Bron: Week 10 – Regression Time Series 2 (read_csv met parse_dates)
# Prompt 1: Portfolio 2 CityBike notebook (klasse DataLoader, clean)

class DataLoader:
    """Leest de bestanden van de Kaggle-competitie in en schoont ze op."""

    def __init__(self, folder='.'):
        self.folder = folder

    def load(self, file_name):
        """Lees een CSV-bestand in met 'date_hour' als datetime."""
        path = os.path.join(self.folder, file_name)
        return pd.read_csv(path, parse_dates=['date_hour'])

    @staticmethod
    def clean(data, outlier_days):
        """Schoon de data op (onderbouwing: paragraaf 1.4 en 1.9).

        - sorteren op tijd en dubbele uren verwijderen;
        - luchtvochtigheid 0 (meetfout) vervangen door interpolatie;
        - uitzonderlijke dagen (outliers) verwijderen.
        """
        data = data.sort_values('date_hour')
        data = data.drop_duplicates('date_hour').copy()
        data.loc[data['hum'] == 0, 'hum'] = np.nan
        data['hum'] = data['hum'].interpolate()
        days = data['date_hour'].dt.normalize()
        data = data[~days.isin(pd.to_datetime(outlier_days))]
        return data.reset_index(drop=True)


loader = DataLoader(DATA_FOLDER)
train_raw = loader.load('train.csv')
test_raw = loader.load('test.csv')
sample_submission = loader.load('sample_submission.csv')
print('train:', train_raw.shape, '| test:', test_raw.shape,
      '| sample_submission:', sample_submission.shape)
train_raw.head()

## 1.3 Datatypes, basisstatistieken en verdeling

Zoals in de DataCamp-cursus van week 9 (Vincent, z.d., hoofdstuk 2) en opdracht 1 van
*Time series analysis 2* (De Haagse Hogeschool, z.d.-g) bekijken we eerst de datatypes, de
basisstatistieken en de verdeling van de target.

In [ ]:
# Bron: Week 10 – Time series analysis 2, opdracht 1 (info en describe)
# Bron: Week 9 – DataCamp Visualizing Time Series Data in Python,
#       hoofdstuk 2 (Summary Statistics and Diagnostics)

# Datatypes en aantal niet-lege waarden per kolom
train_raw.info()

In [ ]:
# Bron: Week 10 – Time series analysis 2, opdracht 1 (info en describe)
# Bron: Week 9 – DataCamp Visualizing Time Series Data in Python,
#       hoofdstuk 2 (Summary Statistics and Diagnostics)

# Basisstatistieken van iedere kolom: trainingsdata en testdata
display(train_raw.describe().round(3))
display(test_raw.describe().round(3))

In [ ]:
# Bron: Week 9 – DataCamp Visualizing Time Series Data in Python,
#       hoofdstuk 2 (histogram en boxplot)

# Verdeling van de target (histogram en boxplot)
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
sns.histplot(train_raw['cnt'], bins=50, ax=axes[0])
axes[0].set(title='Verdeling van cnt', xlabel='cnt (fietsen per uur)')
sns.boxplot(x=train_raw['cnt'], ax=axes[1])
axes[1].set(title='Boxplot van cnt', xlabel='cnt (fietsen per uur)')
plt.tight_layout()
plt.show()

**Toelichting.**

- `date_hour` is een datum-tijd; alle andere kolommen zijn numeriek. Let op: `holiday` is
  **binair** (0/1) en `weathersit` is een **ordinale categorie** (1 = helder … 4 = zware regen),
  ook al zijn ze als getal opgeslagen. Dat bepaalt welke correlatiemaat we gebruiken (1.8) en dat
  we ze niet standaardiseren.
- `temp`, `atemp`, `hum` en `windspeed` zijn genormaliseerd (schaal 0 – 1).
- `cnt` loopt van 1 tot 977. Het gemiddelde (190) ligt ver boven de mediaan (143): de verdeling is
  **scheef naar rechts**, met veel rustige uren en weinig zeer drukke uren (de 'uitschieters' in de
  boxplot zijn gewone spitsuren, geen fouten).
- In de testperiode (december) is het gemiddeld kouder dan in de trainingsdata.

## 1.4 Datakwaliteit

Bij een tijdreeks moeten de tijdstappen **gelijkmatig verdeeld** zijn (Radhakrishnan, 2023b). We
controleren daarom ook op ontbrekende uren, naast missende waarden, dubbele uren en onmogelijke
waarden. De functie `check_quality` is gemaakt met hulp van GenAI (Claude, 2026. Prompt 1:
Portfolio 2 CityBike notebook).

In [ ]:
# Bron: Week 9 – slides Timeseries analyse 1 (tijdstappen gelijkmatig)
# Prompt 1: Portfolio 2 CityBike notebook (functie check_quality)

def check_quality(data):
    """Geef een overzicht van mogelijke datakwaliteitsproblemen."""
    full_range = pd.date_range(data['date_hour'].min(),
                               data['date_hour'].max(), freq='h')
    checks = {
        'missende waarden (totaal)': data.isna().sum().sum(),
        'dubbele uren': data['date_hour'].duplicated().sum(),
        'ontbrekende uren': len(full_range) - data['date_hour'].nunique(),
        'hum == 0': (data['hum'] == 0).sum(),
        'windspeed == 0': (data['windspeed'] == 0).sum(),
        'weathersit == 4': (data['weathersit'] == 4).sum(),
    }
    return pd.Series(checks)


display(pd.DataFrame({'train': check_quality(train_raw),
                      'test': check_quality(test_raw)}))

# Op welke dagen ontbreken de meeste uren?
all_hours = pd.date_range(train_raw['date_hour'].min(),
                          train_raw['date_hour'].max(), freq='h')
missing_hours = all_hours.difference(train_raw['date_hour'])
pd.Series(missing_hours.date).value_counts().head(5)

**Toelichting.**

- Er zijn **geen missende waarden** en **geen dubbele uren**.
- In de trainingsdata **ontbreken 163 uren**, vooral op een paar dagen (o.a. 29–30 oktober 2012 en
  eind januari 2011), waarschijnlijk door storingen of extreem weer. Voor de machine-learning modellen
  is dit geen probleem (elk uur wordt los voorspeld). Voor tijdreeksbewerkingen (FFT, ACF, SARIMAX)
  hebben we wél een regelmatige reeks nodig: daar vullen we ontbrekende uren op met interpolatie.
- **`hum == 0`** (22 keer) is een meetfout: 0% luchtvochtigheid komt in de praktijk niet voor. We
  vervangen deze waarden door lineaire interpolatie.
- **`windspeed == 0`** (±12%) laten we staan: windstilte kan echt voorkomen.
- **`weathersit == 4`** komt maar 3 keer voor; in modellen met categorieën voegen we 3 en 4 samen.
- De testdata is compleet (456 uren) en heeft geen kwaliteitsproblemen.

## 1.5 Tijdselementen toevoegen

Uit `date_hour` halen we tijdselementen, zoals in opdracht 4 van *Regression Time Series 1*
(De Haagse Hogeschool, z.d.-c) en in de seasonal plots van Kaggle-cursus les 3 (Holbrook, z.d.-d):
uur, dag van de week (getal en naam), weeknummer, maand, jaar en dag van het jaar. Daarnaast maken
we `workingday`: maandag t/m vrijdag en geen feestdag.

In [ ]:
# Bron: Week 9 – Regression Time Series 1, opdracht 4 (tijdselementen)
# Bron: Kaggle Time Series, les 3 – Seasonality (Holbrook)

def add_calendar_features(data):
    """Voeg tijdselementen toe die uit 'date_hour' volgen."""
    data = data.copy()
    time = data['date_hour']
    data['hour'] = time.dt.hour
    data['weekday'] = time.dt.dayofweek           # 0 = maandag, 6 = zondag
    data['day_name'] = time.dt.day_name()
    data['week'] = time.dt.isocalendar().week.astype(int)
    data['month'] = time.dt.month
    data['year'] = time.dt.year - 2011            # 0 = 2011, 1 = 2012
    data['dayofyear'] = time.dt.dayofyear
    # werkdag = maandag t/m vrijdag en geen feestdag
    is_weekday = data['weekday'] < 5
    data['workingday'] = (is_weekday & (data['holiday'] == 0)).astype(int)
    return data


eda = add_calendar_features(train_raw)
eda[['date_hour', 'hour', 'weekday', 'day_name', 'week', 'month', 'year',
     'dayofyear', 'workingday']].head(3)